# Weather Forecasting

This notebook trains a weather forecasting model using the same traffic dataset's weather signals.
The project does not yet have a dedicated external weather file, so here we create a realistic forecast pipeline from the available weather columns in the traffic dataset.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import xgboost as xgb

project_dir = Path.cwd().resolve().parent
models_dir = project_dir / 'models'
outputs_dir = project_dir / 'outputs'
models_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

print('Weather forecasting setup complete.')

Weather forecasting setup complete.


In [ ]:
traffic_path = project_dir / 'data' / 'traffic' / 'Metro_Interstate_Traffic_Volume.csv'
if not traffic_path.exists():
    raise FileNotFoundError(f'Missing traffic dataset: {traffic_path}')

weather_df = pd.read_csv(traffic_path)
weather_df['date_time'] = pd.to_datetime(weather_df['date_time'])
weather_df = weather_df.sort_values('date_time').reset_index(drop=True)

weather_df = weather_df.drop_duplicates().dropna(subset=['temp'])
weather_df[['temp', 'rain_1h', 'snow_1h', 'clouds_all']] = weather_df[['temp', 'rain_1h', 'snow_1h', 'clouds_all']].fillna(method='ffill')

weather_df['hour'] = weather_df['date_time'].dt.hour
weather_df['day_of_week'] = weather_df['date_time'].dt.dayofweek
weather_df['month'] = weather_df['date_time'].dt.month
weather_df['weekend'] = (weather_df['day_of_week'] >= 5).astype(int)

print(weather_df.head())
print('Rows:', len(weather_df))
print(f"Dataset date range used for weather training: {weather_df['date_time'].min()} to {weather_df['date_time'].max()}")


  holiday    temp  rain_1h  snow_1h  clouds_all weather_main  \
0     NaN  288.28      0.0      0.0          40       Clouds   
1     NaN  289.36      0.0      0.0          75       Clouds   
2     NaN  289.58      0.0      0.0          90       Clouds   
3     NaN  290.13      0.0      0.0          90       Clouds   
4     NaN  291.14      0.0      0.0          75       Clouds   

  weather_description           date_time  traffic_volume  hour  day_of_week  \
0    scattered clouds 2012-10-02 09:00:00            5545     9            1   
1       broken clouds 2012-10-02 10:00:00            4516    10            1   
2     overcast clouds 2012-10-02 11:00:00            4767    11            1   
3     overcast clouds 2012-10-02 12:00:00            5026    12            1   
4       broken clouds 2012-10-02 13:00:00            4918    13            1   

   month  weekend  
0     10        0  
1     10        0  
2     10        0  
3     10        0  
4     10        0  
Rows: 48187


In [3]:
for lag in [1, 2, 3, 6, 12, 24]:
    weather_df[f'temp_lag_{lag}'] = weather_df['temp'].shift(lag)

weather_df['temp_rolling_mean_3'] = weather_df['temp'].shift(1).rolling(3, min_periods=1).mean()
weather_df['temp_rolling_mean_6'] = weather_df['temp'].shift(1).rolling(6, min_periods=1).mean()
weather_df['temp_rolling_std_6'] = weather_df['temp'].shift(1).rolling(6, min_periods=1).std().fillna(0)

weather_df['temp_target'] = weather_df['temp'].shift(-1)
weather_df = weather_df.dropna(subset=['temp_target'] + [f'temp_lag_{lag}' for lag in [1,2,3,6,12,24]])
weather_df = weather_df.reset_index(drop=True)
print('Prepared rows:', len(weather_df))

Prepared rows: 48162


In [ ]:
FEATURES = [
    'hour', 'day_of_week', 'month', 'weekend',
    'rain_1h', 'snow_1h', 'clouds_all',
    'temp_lag_1', 'temp_lag_2', 'temp_lag_3', 'temp_lag_6', 'temp_lag_12', 'temp_lag_24',
    'temp_rolling_mean_3', 'temp_rolling_mean_6', 'temp_rolling_std_6'
]
TARGET = 'temp_target'

n = len(weather_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train = weather_df.iloc[:train_end]
val = weather_df.iloc[train_end:val_end]
test = weather_df.iloc[val_end:]

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print('Train:', len(train), 'Val:', len(val), 'Test:', len(test))
print(f"Training range: {train['date_time'].min()} to {train['date_time'].max()}")
print(f"Validation range: {val['date_time'].min()} to {val['date_time'].max()}")
print(f"Testing range: {test['date_time'].min()} to {test['date_time'].max()}")

training_metadata = {
    'dataset': 'Metro_Interstate_Traffic_Volume.csv',
    'training_start': str(train['date_time'].min()),
    'training_end': str(train['date_time'].max()),
    'validation_start': str(val['date_time'].min()),
    'validation_end': str(val['date_time'].max()),
    'testing_start': str(test['date_time'].min()),
    'testing_end': str(test['date_time'].max()),
}
joblib.dump(training_metadata, project_dir / 'models' / 'weather_training_metadata.pkl')
print('Saved weather training metadata.')


Train: 33713 Val: 7224 Test: 7225


In [5]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:30s} | MAE: {mae:6.3f} | RMSE: {rmse:6.3f} | R2: {r2:5.3f}')
    return {'model': name, 'MAE': round(mae, 3), 'RMSE': round(rmse, 3), 'R2': round(r2, 3)}

weather_model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse',
    verbosity=0
)
weather_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
weather_pred = weather_model.predict(X_test)
metrics = evaluate('Weather XGBoost', y_test, weather_pred)
print(metrics)

Weather XGBoost                | MAE:  0.733 | RMSE:  0.995 | R2: 0.994
{'model': 'Weather XGBoost', 'MAE': 0.733, 'RMSE': 0.995, 'R2': 0.994}


In [6]:
joblib.dump(weather_model, project_dir / 'models' / 'weather_forecasting_model.pkl')
joblib.dump(FEATURES, project_dir / 'models' / 'weather_features.pkl')

sample = weather_df.iloc[-1][FEATURES].to_dict()
sample_prediction = float(weather_model.predict(pd.DataFrame([sample], columns=FEATURES))[0])
print('Saved weather model and features.')
print('Sample next-hour temperature prediction:', round(sample_prediction, 2))

Saved weather model and features.
Sample next-hour temperature prediction: 281.94
